# 08_twitter_sentiment.ipynb

Updated to match the revised project pipeline: Bitcoin-only Twitter from the large CSV, with analysis based on the saved `bitcoin_twitter_daily.csv` handoff from Notebook 1.

## Cell 1 — Imports & locate Twitter files

In [1]:
import os, json, csv
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Saved outputs from Notebook 1
price = pd.read_csv("price_clean.csv", parse_dates=["date"])
btc_daily = pd.read_csv("bitcoin_twitter_daily.csv", parse_dates=["date"])

# Optional: locate the large raw CSV for provenance / preview only
search_roots = [Path("."), Path("./downloads"), Path("/mnt/data"), Path("/mnt/data/downloads"), Path.home()]
target_name = "engtweetsbtc_vader_sentiment.csv"
matches = []
for root in search_roots:
    try:
        if root.exists():
            matches.extend(root.rglob(target_name))
    except Exception:
        pass

seen = set()
unique_matches = []
for m in matches:
    try:
        resolved = str(m.resolve())
    except Exception:
        resolved = str(m)
    if resolved not in seen:
        seen.add(resolved)
        unique_matches.append(m)

twitter_csv_path = str(unique_matches[0]) if unique_matches else None

print("price shape:", price.shape)
print("bitcoin_twitter_daily shape:", btc_daily.shape)
print("raw twitter csv found:", twitter_csv_path)


price shape: (7594, 8)
bitcoin_twitter_daily shape: (2754, 5)
raw twitter csv found: /home/jovyan/downloads/old_btc_twitter/engtweetsbtc_vader_sentiment.csv


## Cell 2 — Source preview & schema check

In [2]:
if twitter_csv_path is not None:
    with open(twitter_csv_path, "r", encoding="utf-8", errors="ignore") as f:
        sample = f.read(5000)

    try:
        twitter_sep = csv.Sniffer().sniff(sample, delimiters=",;|\t").delimiter
    except Exception:
        twitter_sep = ";"

    preview = pd.read_csv(twitter_csv_path, sep=twitter_sep, nrows=5, low_memory=False)
    print("Using raw CSV:", twitter_csv_path)
    print("Detected separator:", repr(twitter_sep))
    print("Preview columns:", preview.columns.tolist())
    display(preview.head())
else:
    print("Raw CSV not found in workspace. Continuing with Notebook 1 outputs only.")

print("\nbitcoin_twitter_daily columns:", btc_daily.columns.tolist())
display(btc_daily.head())


Using raw CSV: /home/jovyan/downloads/old_btc_twitter/engtweetsbtc_vader_sentiment.csv
Detected separator: ';'
Preview columns: ['origen', 'date', 'username', 'user_fullname', 'user_description', 'user_created', 'user_verified', 'location', 'n_followers', 'n_following', 'user_favourites', 'n_replies', 'n_likes', 'n_retweets', 'hashtags', 'url', 'source', 'is_retweet', 'text', 'tweet_language', 'vader_compound', 'vader_sentiment']


,origen,date,username,user_fullname,user_description,user_created,user_verified,location,n_followers,n_following,...,n_likes,n_retweets,hashtags,url,source,is_retweet,text,tweet_language,vader_compound,vader_sentiment
0,df1,2019-05-27 11:49:18+00:00,bitcointe,Bitcointe,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,Cardano: Digitize Currencies; EOS 6500% ROI; ...,en,-0.1027,NEGATIVE
1,df1,2019-05-27 11:49:06+00:00,3eyedbran,Bran - 3 Eyed Raven,NaN,NaN,NaN,NaN,NaN,NaN,...,2,1,NaN,NaN,NaN,NaN,Another Test tweet that was not caught in the ...,en,0.0000,NEUTRAL
2,df1,2019-05-27 11:49:22+00:00,DetroitCrypto,J. Scardina,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,Current Crypto Prices! \n\nBTC: $8721.99 USD\n...,en,0.0000,NEUTRAL
3,df1,2019-05-27 11:49:25+00:00,evilrobotted,evilrobotted,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,We have been building on the real #bitcoin SV....,en,-0.4767,NEGATIVE
4,df1,2019-05-22 12:42:16+00:00,Cybintelligence,CybIntelligence,NaN,NaN,NaN,NaN,NaN,NaN,...,2,7,NaN,NaN,NaN,NaN,BestMixer has been seized by the Dutch Police ...,en,0.0000,NEUTRAL



bitcoin_twitter_daily columns: ['coin', 'date', 'btc_twitter_sentiment_mean', 'btc_twitter_sentiment_std', 'btc_twitter_tweet_count']


,coin,date,btc_twitter_sentiment_mean,btc_twitter_sentiment_std,btc_twitter_tweet_count
0,Bitcoin,2013-01-01,-0.005627,0.166366,11
1,Bitcoin,2013-01-02,0.020000,0.082462,17
2,Bitcoin,2013-01-03,0.102525,0.189335,24
3,Bitcoin,2013-01-04,-0.006246,0.208878,28
4,Bitcoin,2013-01-05,0.043478,0.232035,27


## Cell 3 — Build Bitcoin-only sentiment analysis frame

In [3]:
btc_price = price[price["coin"] == "Bitcoin"].copy().sort_values("date")
btc_price["target"] = (btc_price["close"].shift(-1) > btc_price["close"]).astype("Int64")

btc_merged = btc_price.merge(btc_daily, on=["coin", "date"], how="left")
sent_cols = [
    "btc_twitter_sentiment_mean",
    "btc_twitter_sentiment_std",
    "btc_twitter_tweet_count",
]
btc_merged[sent_cols] = btc_merged[sent_cols].fillna(0)

print("btc_merged shape:", btc_merged.shape)
print("Bitcoin date range:", btc_merged["date"].min().date(), "to", btc_merged["date"].max().date())
display(btc_merged.head())


btc_merged shape: (4056, 12)
Bitcoin date range: 2010-07-18 to 2021-08-24


,coin,date,open,high,low,close,volume,pct_change_raw,target,btc_twitter_sentiment_mean,btc_twitter_sentiment_std,btc_twitter_tweet_count
0,Bitcoin,2010-07-18,0.0,0.1,0.1,0.1,80.0,0.0,0,0.0,0.0,0.0
1,Bitcoin,2010-07-19,0.1,0.1,0.1,0.1,570.0,0.0,0,0.0,0.0,0.0
2,Bitcoin,2010-07-20,0.1,0.1,0.1,0.1,260.0,0.0,0,0.0,0.0,0.0
3,Bitcoin,2010-07-21,0.1,0.1,0.1,0.1,580.0,0.0,0,0.0,0.0,0.0
4,Bitcoin,2010-07-22,0.1,0.1,0.1,0.1,2160.0,0.0,0,0.0,0.0,0.0


## Cell 4 — Coverage and summary statistics

In [4]:
covered_days = int((btc_merged["btc_twitter_tweet_count"] > 0).sum())
coverage_ratio = covered_days / len(btc_merged) if len(btc_merged) else 0

print("Bitcoin Twitter daily rows:", len(btc_daily))
print("Bitcoin price rows:", len(btc_merged))
print("Covered Bitcoin price days:", covered_days)
print("Coverage ratio:", f"{coverage_ratio:.2%}")

print("\nSentiment summary on covered days:")
display(
    btc_merged.loc[btc_merged["btc_twitter_tweet_count"] > 0, sent_cols]
    .describe()
)


Bitcoin Twitter daily rows: 2754
Bitcoin price rows: 4056
Covered Bitcoin price days: 2754
Coverage ratio: 67.90%

Sentiment summary on covered days:


,btc_twitter_sentiment_mean,btc_twitter_sentiment_std,btc_twitter_tweet_count
count,2754.000000,2754.000000,2754.000000
mean,0.077335,0.227629,5305.605301
std,0.072004,0.096137,11422.116897
min,-0.080436,0.000000,11.000000
25%,0.028035,0.166768,333.000000
50%,0.074326,0.213303,526.500000
75%,0.110129,0.267804,700.750000
max,0.600473,0.451926,102822.000000


## Cell 5 — Visualise tweet volume and mean sentiment

In [5]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

axes[0].bar(
    btc_daily["date"],
    btc_daily["btc_twitter_tweet_count"],
    color="#93c5fd",
    alpha=0.8,
    width=1,
)
axes[0].set_title("Bitcoin Twitter Daily Tweet Count")
axes[0].set_ylabel("Tweets/day")
axes[0].grid(alpha=0.2)

axes[1].plot(
    btc_daily["date"],
    btc_daily["btc_twitter_sentiment_mean"],
    color="#1d4ed8",
    linewidth=1.2,
)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title("Bitcoin Twitter Daily Mean Sentiment")
axes[1].set_ylabel("Mean VADER compound")
axes[1].grid(alpha=0.2)

plt.suptitle("Bitcoin Twitter Sentiment Overview", fontsize=14)
plt.tight_layout()
plt.savefig("bitcoin_twitter_overview.png", dpi=150)
plt.show()


## Cell 6 — Sentiment vs next-day direction

In [6]:
corr_frame = btc_merged[
    ["target", "btc_twitter_sentiment_mean", "btc_twitter_sentiment_std", "btc_twitter_tweet_count"]
].dropna()

print("=== Bitcoin sentiment vs next-day direction correlation ===")
print(corr_frame.corr()[["target"]])

covered = btc_merged[btc_merged["btc_twitter_tweet_count"] > 0].copy()
if len(covered) > 0:
    up_days = covered.loc[covered["target"] == 1, "btc_twitter_sentiment_mean"]
    down_days = covered.loc[covered["target"] == 0, "btc_twitter_sentiment_mean"]
    print("\nCovered days only:")
    print("Mean sentiment on Up days:  ", round(up_days.mean(), 6) if len(up_days) else "n/a")
    print("Mean sentiment on Down days:", round(down_days.mean(), 6) if len(down_days) else "n/a")
else:
    print("No covered Bitcoin days available.")


=== Bitcoin sentiment vs next-day direction correlation ===
                              target
target                      1.000000
btc_twitter_sentiment_mean  0.029376
btc_twitter_sentiment_std   0.086180
btc_twitter_tweet_count     0.008065

Covered days only:
Mean sentiment on Up days:   0.073156
Mean sentiment on Down days: 0.082134


## Cell 7 — Save analysis-friendly output

In [7]:
btc_merged.to_csv("bitcoin_twitter_analysis_frame.csv", index=False)
print("✅ Saved bitcoin_twitter_analysis_frame.csv")
print("✅ Existing feature-engineering handoff remains bitcoin_twitter_daily.csv")


✅ Saved bitcoin_twitter_analysis_frame.csv
✅ Existing feature-engineering handoff remains bitcoin_twitter_daily.csv
